In [1]:
import os
import pandas as pd
import numpy as np

# A base completa original (train_ver2.csv) possui 2.2 GB.
# Para viabilizar a execução sem estouro de memória e permitir o versionamento no Git,
# disponibilizamos a subamostra oficial de 30.000 clientes compactada (gzip).

caminho_amostra_gz = 'santander_30k_sample.csv.gz'

if os.path.exists(caminho_amostra_gz):
    print("Carregando amostra otimizada de 30.000 clientes...")
    df = pd.read_csv(caminho_amostra_gz, compression='gzip', low_memory=False)
    print(f"Sucesso! Registros carregados: {df.shape[0]:,} | Clientes: {df['ncodpers'].nunique():,}")
else:
    # Fluxo alternativo: Caso queira reprocessar o arquivo bruto do Kaggle
    caminho_bruto = 'train_ver2.csv'

    if not os.path.exists(caminho_bruto):
        raise FileNotFoundError(
            f"Arquivo '{caminho_amostra_gz}' não encontrado. "
            "Certifique-se de clonar o repositório completo ou disponibilizar 'train_ver2.csv'."
        )

    print("Mapeando IDs únicos a partir do arquivo bruto...")
    df_ids = pd.read_csv(caminho_bruto, usecols=['ncodpers'], dtype={'ncodpers': np.int32})
    clientes_unicos = df_ids['ncodpers'].unique()
    del df_ids

    np.random.seed(42)
    clientes_amostra = set(np.random.choice(clientes_unicos, size=30000, replace=False))

    print("Extraindo fatias temporais dos 30.000 clientes selecionados...")
    chunks_filtrados = [
        chunk[chunk['ncodpers'].isin(clientes_amostra)]
        for chunk in pd.read_csv(caminho_bruto, chunksize=1_000_000, low_memory=False)
    ]
    df = pd.concat(chunks_filtrados, ignore_index=True)
    del chunks_filtrados

    df.to_csv(caminho_amostra_gz, index=False, compression='gzip')
    print("Amostra extraída e compactada com sucesso!")

Carregando amostra otimizada de 30.000 clientes...
Sucesso! Registros carregados: 427,751 | Clientes: 30,000
